In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch

from tsfm_lens.chronos.circuitlens import CircuitLensChronos
from tsfm_lens.utils import (
    make_ensemble_from_arrow_dir,
    plot_3d_and_univariate,
    plot_multi_3d_and_univariate,
    set_seed,
)

In [ ]:
DEFAULT_COLORS = list(plt.get_cmap("tab20").colors)  # type: ignore
print(f"{len(DEFAULT_COLORS)} colors")

In [ ]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
WORK_DIR = os.getenv("WORK", "/work")
DATA_DIR = os.path.join(WORK_DIR, "data")
plot_save_dir = os.path.join("../figs", "logit_attribution")
os.makedirs(plot_save_dir, exist_ok=True)

### Load Data

In [ ]:
split_name = "base40"
split_dir = os.path.join(DATA_DIR, split_name)

system_name = "Lorenz"
subsample_interval = 2

ensemble = make_ensemble_from_arrow_dir(split_dir, dyst_names_lst=[system_name], num_samples=1)

In [ ]:
context_length = 512
context_start_time = 500
context_end_time = context_start_time + context_length

In [ ]:
# Prepare input series
sample_idx = 0
# get sample sample_idx of Lorenz
dyst_coords = torch.tensor(ensemble["Lorenz"][sample_idx, :, :])
dyst_coords = dyst_coords[:, ::subsample_interval]
context = dyst_coords[:, context_start_time:context_end_time]

print(f"dyst_coords shape: {dyst_coords.shape}")
print(f"context shape: {context.shape}")

In [ ]:
plot_3d_and_univariate(context, custom_colors=["black"])

### Circuit Analysis

In [ ]:
pipeline = CircuitLensChronos.from_pretrained("amazon/chronos-t5-base", device_map=device, torch_dtype=torch.bfloat16)
num_layers = pipeline.model.model.config.num_decoder_layers
num_heads = pipeline.model.model.config.num_heads

print(f"num_layers: {num_layers}, num_heads: {num_heads}")

In [ ]:
prediction_length = 64

if prediction_length != 64:
    warnings.warn(
        "Prediction length is not 64, which is not supported for the current implementation, since we don't support accumulating read stream outputs yet"
    )

In [ ]:
future_vals = dyst_coords[:, context_end_time : context_end_time + prediction_length]
print(f"future_vals shape: {future_vals.shape}")

In [ ]:
rseed = 123
set_seed(rseed)

In [ ]:
# List of layer indices to hook into and read the residual stream, from the decoder
stream_positions = [i for i in range(num_layers)]

# List of (layer_idx, head_idx) tuples to ablate.
# NOTE: we are currently keeping all heads, but knocking out a range of layers specifically the cross-attention layers of the decoder
head_ablations = [(i, j) for i in range(4, 9) for j in range(num_heads)]
print(f"Abalating (layer_idx, head_idx): {head_ablations}")
print(f"Reading stream positions: {stream_positions}")

# add hooks to the pipeline
pipeline.add_read_stream_hooks(stream_positions)
pipeline.add_head_ablation_hooks(head_ablations)

predictions, full_outputs = pipeline.predict_with_full_outputs(
    context=context,
    prediction_length=prediction_length,
    num_samples=1,
    do_sample=False,  # greedy decoding
    use_cache=False,
    return_dict_in_generate=True,
    output_scores=True,
    limit_prediction_length=False,
)

In [ ]:
predictions.shape

In [ ]:
context.shape

In [ ]:
len(full_outputs)

In [ ]:
selected_dim = 0

plt.figure(figsize=(6, 2))
ctx = context[selected_dim].cpu().numpy() if hasattr(context[selected_dim], "cpu") else context[selected_dim]
# plot ground truth future values as dotted black line
future_vals_timesteps = np.arange(context_length, context_length + prediction_length)
plt.plot(
    future_vals_timesteps,
    future_vals[selected_dim],
    color="black",
    linewidth=1,
    linestyle="--",
    label="Ground Truth",
)
preds = predictions[selected_dim]  # NOTE: treat batch dimension as the data dimension
n, L = preds.shape

plt.plot(np.arange(len(ctx)), ctx, color="black", linewidth=1, label="Context")
for i in range(n):
    plt.plot(
        np.arange(len(ctx), len(ctx) + L),
        preds[i],
        color=DEFAULT_COLORS[i],
        linewidth=1,
        # label=f"Prediction {i + 1}" if i == 0 else None,
    )
plt.xlabel("Timestep", fontweight="bold")
# plt.legend(frameon=True, loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(plot_save_dir, f"predictions_{system_name}_dim{selected_dim}.pdf"))
plt.show()

In [ ]:
def sample_tokens(residual: torch.Tensor, do_sample: bool = False) -> torch.Tensor:
    """
    Sample tokens from residual stream

    Args:
        residual: Residual stream tensor of shape (batch_size, seq_len, d_model)
        do_sample: Whether to use sampling instead of greedy decoding (not implemented)

    Returns:
        Token indices of shape (batch_size, seq_len) from greedy decoding
    """
    if do_sample:
        raise NotImplementedError

    logits = pipeline.unembed_residual(residual)
    return torch.argmax(logits, dim=-1)

In [ ]:
future = dyst_coords[:, context_length : context_length + prediction_length]
context_tokens, context_attention_mask, context_scale = pipeline.tokenizer.context_input_transform(context)
print(f"context_tokens shape: {context_tokens.shape}")
print(f"context_attention_mask shape: {context_attention_mask.shape}")
print(f"context_scale shape: {context_scale.shape}")

In [ ]:
future_tokens, future_attention_mask, future_scale = pipeline.tokenizer._input_transform(future, context_scale)
print(f"future_tokens shape: {future_tokens.shape}")
print(f"future_attention_mask shape: {future_attention_mask.shape}")
print(f"future_scale shape: {future_scale.shape}")

### Layer Streams

In [ ]:
tokens_per_layer_stream = {}
for i in range(num_layers):
    sampled_tokens = sample_tokens(pipeline.read_stream_outputs[i]).float().cpu().numpy()
    print(f"layer {i} sampled_tokens shape: {sampled_tokens.shape}")
    tokens_per_layer_stream[f"layer {i}"] = sampled_tokens

In [ ]:
# combine tokens from selected layers streams with {"ground truth": future_tokens}
sel_layer_stream_tokens = {
    **tokens_per_layer_stream,
    "ground truth": future_tokens,
}
for key, value in sel_layer_stream_tokens.items():
    print(f"{key}: {value.shape}")

In [ ]:
custom_colors = plt.get_cmap("Blues")(np.linspace(0.2, 0.9, num_layers))
colors_dict = {
    "ground truth": "black",
    **{f"layer {i}": custom_colors[i] for i in range(num_layers)},
}

plot_multi_3d_and_univariate(sel_layer_stream_tokens, colors_dict, figsize=(6, 12), use_colorbar_legend=True)

In [ ]:
# pipeline.model.model.encoder

### Visualize Output Scores

TODO: fix the normalization wrt the scales

In [ ]:
selected_dim = 2

all_scores = np.concatenate(
    [
        np.array([s[selected_dim].detach().cpu().numpy() for s in full_outputs[i].scores])
        for i in range(len(full_outputs))
    ],
    axis=0,
)
print(all_scores.shape)

In [ ]:
# Concise heatmap plot, centered at 0
token_id_range = (0, 4096)
time_step_range = (0, prediction_length)

scores = all_scores[time_step_range[0] : time_step_range[1], token_id_range[0] : token_id_range[1]].T
vmin = np.min(scores[scores != -np.inf]) if np.isinf(scores.min()) else scores.min()
vmax = np.max(scores[scores != np.inf]) if np.isinf(scores.max()) else scores.max()

# Center the colormap at 0
abs_max = max(abs(vmin), abs(vmax))
vmin_centered = -abs_max
vmax_centered = abs_max

plt.figure(figsize=(12, 8))
plt.imshow(scores, aspect="auto", cmap="RdBu", vmin=vmin_centered, vmax=vmax_centered)
plt.colorbar(label="Score Values")
plt.ylabel("Token ID", fontweight="bold")
plt.xlabel("Time Step", fontweight="bold")
plt.title("Scores", fontweight="bold")
plt.show()